## Реккурентные нейросети. GRU

- [Hidden state](#hidden-state)
- [Bidirectional RNN](#bidirectional-rnn)
- [GRU](#gru)

_Пример. Реккурентная нейросеть, many-to-many:_

<img src="imgs/img_1.png" width="550px"/>

- Все блоки одинаковые с общими параметрами


_Пример. Реккурентная нейросеть, many-to-one:_

<img src="imgs/img_2.png" width="550px"/>


### Hidden state

Чтобы хранить информацию о предыдущих токенах, мы вводим понятие внутренней памяти или **скрытого состояния** (hidden state, векторы $h_i$)

- Обновление скрытого состояния: 

$\qquad h_i = \text{tanh}(h_{i-1}W_1 + x_i W_2)$

- Предсказание по скрытому состоянию:

$\qquad y_i = h_iW_3$

> _Замечание._ Веса $W_k$ одинаковы на всех итерациях, то есть вы можете представлять себе, что очередные $x_i$ и $h_{i-1}$ подаются на вход одного и того же слоя, зацикленного на себе


_Пример. Глубокая реккурентая нейросеть_

<img src="imgs/img_3.png" width="550px"/>

- Несколько рекуррентных слоев:
    - первый слой RNN будет принимать на вход исходную последовательность 
    - вторая RNN — выходы первой сети 
    - третья — выходы второй и т.д.

### Bidirectional RNN

Стандартная RNN учитывает только предыдущий контекст. Но ведь слово в
предложении связано не только с предыдущими, но и с последующими словами. В
таких случаях имеет смысл использовать двунаправленную рекуррентную сеть
(bidirectional RNN, BRNN).

Как следует из названия, в bidirectional RNN есть две рекуррентных подсети:
прямая (_forward_, токены в нее подаются от первого к последнему) и обратная
(_backward_, токены подаются в обраттном порядке).


_Пример. Bidirectional RNN_

<img src="imgs/img_4.png" width="550px"/>

> _Замечание._ Формула для $y_i$ может быть и другой. Например, выходы обеих
рекуррентных сетей могут агрегироваться путем усреднения, или суммирования,
или любым другим способом

## GRU

> Gated Recurrent Unit был предложен в [статье](https://arxiv.org/pdf/1406.1078v3) Cho et al. в 2014 году

- Объединяет _input gate_ и _forget gate_ в один **update gate** 
- Устраняет разделение внутренней информации блока на _hidden_ и _cell state_ 

_Пример. Общий вид GRU-блока_

<img src="imgs/img_5.png" width="350px"/>


### Update gate

Будем забывать только те значения, которые собираемся обновить. 

$$\text{update gate:} \qquad z_t = \sigma(h_{t-1}W_1^z + x_tW_2^z + b_z)$$

### Reset gate

Новый тип гейта, который появляется в GRU. Он определяет, какую долю информации из $h_{t−1}$ с прошлого шага надо "сбросить", инициализировать заново.

$$\text{reset gate:} \qquad r_t = \sigma(h_{t-1}W_1^r + x_tW_2^r + b_r)$$

Далее вычисляем **потенциальное обновление** для скрытого слоя:

$$\tilde h_t = \text{tanh}\Big[ (r_t \odot h_{t-1})W_1^h + x_tW_2^h + b_h\Big]$$

И решаем, что из старого забыть, а что из нового добавить:

$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde h_t$$

Выход:

$$y_t = h_t$$

**Интерпретация**

- _GRU решает проблему долгосрочных зависимостей_ через адаптивное смешивание состояний
    - _ворота обновления_ ($z_t$): сколько информации сохранить из прошлого?
    - _ворота сброса_ ($r_t$): какую часть истории игнорировать при генерации нового состояния?
- _Ключевые упрощения относительно LSTM_
    - нет отдельной ячейки памяти ($c_t$ и $h_t$ объединены)
    - отсутствуют ворота выхода ($o_t$)
    - динамическое обновление через линейную интерполяцию вместо сложных взаимодействий
- _Эффективность_
    - меньше параметров $\rightarrow$ быстрее обучение, меньше риск переобучения
    - часто превосходит LSTM на коротких и средних последовательностях